In [0]:
%python
# Task 5: Data Quality & Reconciliation (PySpark)
# Single tabular report: category | check_name | severity | detail | status
# Pipeline stops (raises) only if any CRITICAL check fails.

from pyspark.sql import functions as F
from pyspark.sql import SparkSession
import pandas as pd

# Parameter: catalog name (editable via the 'catalog_name' widget / job parameter)
dbutils.widgets.text("catalog_name", "dev")
catalog_name = dbutils.widgets.get("catalog_name")

spark = SparkSession.builder.getOrCreate()

silver_schema = f"{catalog_name}.silver"
gold_schema = f"{catalog_name}.gold"

results = []   # rows of (category, check_name, severity, detail, status)
critical_failures = []

def record(category, check_name, severity, detail, passed):
    status = "PASS" if passed else "FAIL"
    results.append((category, check_name, severity, str(detail), status))
    print(f"[{status}] {category} | {check_name} ({severity}): {detail}")
    if not passed and severity == "CRITICAL":
        critical_failures.append(f"{category} | {check_name}: {detail}")

# 1. Confirm all eight TPCH source tables exist in silver
cat = "1_silver_tables"
silver_tables = ["customer", "supplier", "nation", "region", "part", "partsupp", "orders", "lineitem"]
missing = []
for tbl in silver_tables:
    try:
        spark.catalog.getTable(f"{silver_schema}.{tbl}")
    except Exception:
        missing.append(tbl)
record(cat, "silver ingested", "CRITICAL",
       "All 8 present" if not missing else f"Missing: {missing}",
       not missing)

# 2. PK uniqueness checks (gold dims)
cat = "2_pk_uniqueness"
for dim_name, pk_col in [
    ("dim_customer", "customer_id"),
    ("dim_product",  "product_id"),
    ("dim_supplier", "supplier_id"),
    ("dim_geography", "nation_id")]:
    df = spark.table(f"{gold_schema}.{dim_name}")
    total = df.count()
    unique = df.select(pk_col).distinct().count()
    passed = unique == total and total > 0
    record(cat, f"gold.{dim_name} PK unique", "CRITICAL",
           f"unique={unique}, total={total}", passed)

# 3. Fact table null surrogate keys
cat = "3_fact_integrity"
fact = spark.table(f"{gold_schema}.fact_sales")
null_fact_keys = fact.filter(
    F.col("order_id").isNull() | F.col("customer_id").isNull() |
    F.col("product_id").isNull() | F.col("supplier_id").isNull()
).count()
record(cat, "fact_sales null surrogate keys", "CRITICAL", null_fact_keys, null_fact_keys == 0)

# 4. Numeric rules
invalid_measures = fact.filter(
    (F.col("extended_price") < 0) |
    (F.col("quantity") < 0) |
    (F.col("discount") < 0) |
    (F.col("tax") < 0)
).count()
record(cat, "negative numeric measures", "CRITICAL", invalid_measures, invalid_measures == 0)

# 5. Gold vs Silver reconciliation
cat = "4_reconciliation"
gold_revenue = fact.select((F.col("extended_price") * (1 - F.col("discount"))).alias("net_revenue")).agg(F.sum("net_revenue")).collect()[0][0] or 0
silver = spark.table(f"{silver_schema}.lineitem")
silver_revenue = silver.select((F.col("l_extendedprice") * (1 - F.col("l_discount"))).alias("net_revenue")).agg(F.sum("net_revenue")).collect()[0][0] or 0
rev_match = abs(gold_revenue - silver_revenue) <= 1e-6
record(cat, "total net revenue gold vs silver", "CRITICAL",
       f"gold={gold_revenue:,.6f}, silver={silver_revenue:,.6f}, diff={gold_revenue - silver_revenue:,.6f}",
       rev_match)

# 6. Quarantined rows (informational; missing table flagged as WARN)
cat = "5_quarantine"
for q_tbl, lbl in [
    ("quarantine_customer", "customers"),
    ("quarantine_supplier", "suppliers"),
    ("quarantine_lineitem", "lineitem")]:
    try:
        q_count = spark.table(f"{silver_schema}.{q_tbl}").count()
        record(cat, f"quarantine_{lbl} present", "WARN", f"{q_count} rows", True)
    except Exception:
        record(cat, f"quarantine_{lbl} present", "WARN", "<table missing>", False)

# 7. Tabular summary of all checks
report_df = pd.DataFrame(results, columns=["category", "check_name", "severity", "detail", "status"])
display(report_df)

# 8. Persist the report to <catalog>.ops.data_quality_reconciliation (catalog parameterized)
report_target = f"{catalog_name}.ops.data_quality_reconciliation"
report_sdf = (spark.createDataFrame(report_df)
              .withColumn("run_ts", F.current_timestamp()))
report_sdf.write.format("delta").option('mergeSchema', 'true').mode("append").saveAsTable(report_target)
print(f"Report saved to {report_target}")

# 9. Fail pipeline only if any CRITICAL check failed
if critical_failures:
    raise Exception("Pipeline STOP: CRITICAL data quality failures\n" + "\n".join(critical_failures))
else:
    print("All CRITICAL data quality checks passed.")
